# Train an RVC v2 voice model on Kaggle
# Обучение RVC v2 в Kaggle

This notebook trains an **RVC v2, 48 kHz, F0-conditioned** voice model on a free Kaggle GPU and leaves the finished `.pth` + `.index` in `/kaggle/working`, where you download them from the **Output** tab. The result installs into [RVC-WebUI-MacOS](https://github.com/ltillegal/rvc-macos-sequoia) with the built-in ZIP installer.

Этот ноутбук обучает голосовую модель **RVC v2, 48 кГц, с F0** на бесплатном GPU в Kaggle и складывает готовые `.pth` + `.index` в `/kaggle/working`, откуда они скачиваются на вкладке **Output**. Файлы ставятся в RVC-WebUI-MacOS через встроенный ZIP-установщик.

> **Important / Важно:** RVC and Seed-VC are different architectures. An RVC `.pth` does **not** work in Seed-VC, and a Seed-VC checkpoint does not work here.
>
> RVC и Seed-VC — разные архитектуры. RVC-модель **не** открывается в Seed-VC, и наоборот.

## Before you run / Перед запуском

1. **Settings → Accelerator → GPU T4 x2** (P100 also works) and **Settings → Internet → On**. Without internet the install step cannot run.
2. **Settings → Accelerator** and **Settings → Internet** are in the right sidebar of the notebook editor.
   / Эти два переключателя — в правой панели редактора ноутбука.
3. Upload your audio as a **private Kaggle Dataset**: *Datasets → New Dataset → drag the files → Create*. One long file is fine. **No dataset?** Put a direct link into `AUDIO_URL` in cell 1 instead - the notebook downloads it itself.\n   / **Нет датасета?** Вставьте прямую ссылку в `AUDIO_URL` в ячейке 1, ноутбук скачает файл сам.
4. In the notebook: **Add Input → Your Datasets →** that dataset. It appears under `/kaggle/input/<name>`.
   / **Add Input → Your Datasets →** ваш датасет, он появится в `/kaggle/input/<имя>`.
5. Run the notebook.

## Two ways to run / Два способа запуска

- **Run all** in the interactive session — good for a first try, but the tab must stay open.
- **Save Version → Save & Run All (Commit)** — recommended for real training: it runs headless on Kaggle's side for up to 9–12 hours, keeps the output automatically, and does not care about your browser.
  / Для реального обучения лучше **Save Version → Save & Run All (Commit)**: считается на сервере до 9–12 часов, вывод сохраняется сам, браузер можно закрыть.

**Quota / Квота** — free accounts get about 30 GPU-hours per week, and one session is capped at 12 hours. Check *Settings → GPU quota* before starting.

## Before you start / Перед началом

**Dataset / Датасет**

- 10–30 minutes of one speaker is the sweet spot. 4 minutes works but the timbre will be less stable.
- Clean speech: no music, no echo, no second voice, no clipping, even level.
- WAV / MP3 / FLAC / M4A / OGG all work. Everything is converted to mono 48 kHz automatically.
- Do not over-denoise: aggressive cleanup becomes part of the model ("metallic" voice).
- 10–30 минут одного голоса — оптимум. 4 минуты тоже обучатся, но тембр будет менее стабильным. Без музыки, эха, второго голоса и клиппинга. Агрессивный денойз вредит.

**Where things live / Что где лежит**

| Path | What | Survivability |
| --- | --- | --- |
| `/kaggle/input/<dataset>` | your audio, read-only | permanent |
| `/kaggle/temp/rvc` | venv, repository, weights, preprocessed audio | scratch, fast, ~20 GB |
| `/kaggle/working/rvc/<name>` | finished `.pth`, `.index`, checkpoints, ZIP | saved with the version |

`/kaggle/temp` is not saved in the output — only `/kaggle/working` is. Nothing there matters except the last three files.
`/kaggle/temp` в вывод не попадает — сохраняется только `/kaggle/working`, а из него важны только готовые файлы.

**Time / Время** — ~10–20 min installing and preprocessing + ~40–90 min for 200 epochs on a T4.

**Session limits / Лимиты сессии** — if the interactive session drops, run everything again: cell 4 reinstalls (5–10 min), preprocessing is redone, training resumes from the checkpoints in `/kaggle/working`.

In [ ]:
#@title 1. Settings / Настройки { display-mode: "form" }
#@markdown Audio comes from either an attached dataset (`Add Input`) or a direct link in `AUDIO_URL`.
#@markdown / Аудио берётся из подключённого датасета (`Add Input`) или по прямой ссылке в `AUDIO_URL`.

EXPERIMENT   = "mymodel"  #@param {type:"string"}
DATASET_DIR  = ""  #@param {type:"string"}
AUDIO_URL    = ""  #@param {type:"string"}
SAMPLE_RATE  = "48k"  #@param ["48k", "40k", "32k"]
TOTAL_EPOCHS = 200  #@param {type:"integer"}
SAVE_EVERY   = 25  #@param {type:"integer"}
BATCH_SIZE   = 8  #@param {type:"integer"}
SPEAKER_ID   = 0  #@param {type:"integer"}
REPO_URL     = "https://github.com/ltillegal/rvc-macos-sequoia.git"  #@param {type:"string"}

import os

WORK        = "/kaggle/temp/rvc"          # scratch, not saved in the output
OUTPUT_ROOT = "/kaggle/working/rvc"       # saved with the version
REPO_DIR    = os.path.join(WORK, "repo")
VENV_DIR    = os.path.join(WORK, ".venv")
PY          = os.path.join(VENV_DIR, "bin", "python")
LOCAL_DATA  = os.path.join(WORK, "dataset")
TRAIN_DATA  = os.path.join(WORK, "dataset_48k")
LOG_DIR     = os.path.join(REPO_DIR, "logs", EXPERIMENT)
WEIGHTS_DIR = os.path.join(REPO_DIR, "assets", "weights")
OUTPUT_DIR  = os.path.join(OUTPUT_ROOT, EXPERIMENT)
CKPT_BACKUP = os.path.join(OUTPUT_DIR, "checkpoints")

SR      = {"32k": 32000, "40k": 40000, "48k": 48000}[SAMPLE_RATE]
VERSION = "v2" if SAMPLE_RATE == "48k" else "v1"

if VERSION == "v2":
    PRETRAIN_G = os.path.join(REPO_DIR, "assets", "pretrained_v2", "f0G48k.pth")
    PRETRAIN_D = os.path.join(REPO_DIR, "assets", "pretrained_v2", "f0D48k.pth")
else:
    PRETRAIN_G = os.path.join(REPO_DIR, "assets", "pretrained", f"f0G{SAMPLE_RATE}.pth")
    PRETRAIN_D = os.path.join(REPO_DIR, "assets", "pretrained", f"f0D{SAMPLE_RATE}.pth")

os.makedirs(WORK, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Experiment : {EXPERIMENT}")
print(f"Sample rate: {SAMPLE_RATE} (RVC {VERSION}) / Частота: {SAMPLE_RATE}")
print(f"Epochs     : {TOTAL_EPOCHS} (checkpoint every {SAVE_EVERY})")
print(f"Scratch    : {WORK}")
print(f"Output     : {OUTPUT_DIR}")
print(f"Dataset    : {DATASET_DIR or '(auto: first dataset under /kaggle/input)'}")
print(f"Audio URL  : {AUDIO_URL or '(none)'}")

In [ ]:
#@title 2. Check the GPU / Проверка GPU
import subprocess
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout or "NO GPU FOUND")
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)

In [ ]:
#@title 3. Where the audio comes from / Откуда взять аудио
# Three ways, tried in this order:
#   1. a dataset attached with "Add Input" (read-only at /kaggle/input),
#   2. an archive (.zip) inside that dataset - unpacked to the fast disk,
#   3. a direct link in AUDIO_URL (cell 1): Google Drive "anyone with the link",
#      Dropbox, or any https URL that returns the file itself.
# Три способа, по порядку: датасет через "Add Input"; архив .zip внутри него;
# прямая ссылка в AUDIO_URL (ячейка 1).
import os, subprocess, sys, zipfile

EXTS = (
    ".wav", ".mp3", ".flac", ".m4a", ".ogg", ".aif", ".aiff", ".opus",
    ".wma", ".aac", ".mp4", ".mkv", ".webm", ".caf", ".au",
)
ARCHIVES = (".zip", ".tar", ".tar.gz", ".tgz")


def audio_files(path):
    return [
        os.path.join(cur, name)
        for cur, _dirs, names in os.walk(path)
        for name in sorted(names)
        if name.lower().endswith(EXTS)
    ]


def unpack(archives, dest):
    os.makedirs(dest, exist_ok=True)
    for path in archives:
        print(f"unpacking / распаковка: {path}", flush=True)
        if path.lower().endswith(".zip"):
            with zipfile.ZipFile(path) as z:
                z.extractall(dest)
        else:
            import tarfile
            with tarfile.open(path) as t:
                t.extractall(dest)
    return audio_files(dest)


def listing(path):
    for cur, dirs, names in os.walk(path):
        dirs.sort()
        for name in sorted(names):
            full = os.path.join(cur, name)
            print(f"   {os.path.getsize(full) / 1024 / 1024:8.1f} MB  {full}")


# ---- 1. attached datasets --------------------------------------------------
root = "/kaggle/input"
print("=== /kaggle/input ===")
found = []
if not os.path.isdir(root) or not os.listdir(root):
    print("   empty: nothing attached / пусто: датасет не подключён")
else:
    for name in sorted(os.listdir(root)):
        path = os.path.join(root, name)
        files = audio_files(path) if os.path.isdir(path) else []
        print(f"   {name:38s} {len(files):4d} audio file(s)")
        if files and not found:
            DATASET_DIR, found = path, files

# ---- 2. archive instead of plain audio -------------------------------------
if not found:
    archives = [
        os.path.join(cur, name)
        for cur, _dirs, names in os.walk(root)
        for name in sorted(names)
        if name.lower().endswith(ARCHIVES)
    ] if os.path.isdir(root) else []
    if archives:
        unpacked = os.path.join(WORK, "unpacked")
        found = unpack(archives, unpacked)
        if found:
            DATASET_DIR = unpacked

# ---- 3. direct link --------------------------------------------------------
if not found and AUDIO_URL:
    dest = os.path.join(WORK, "downloaded")
    os.makedirs(dest, exist_ok=True)
    print(f"downloading / скачиваем: {AUDIO_URL}", flush=True)
    if "drive.google.com" in AUDIO_URL:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
        subprocess.run(
            [sys.executable, "-c",
             "import sys, gdown; gdown.download(sys.argv[1], quiet=False, fuzzy=True)",
             AUDIO_URL],
            cwd=dest, check=True,
        )
    else:
        import urllib.request
        name = os.path.basename(AUDIO_URL.split("?")[0]) or "audio"
        request = urllib.request.Request(AUDIO_URL, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(request) as r, open(os.path.join(dest, name), "wb") as f:
            f.write(r.read())
    found = audio_files(dest)
    if not found:
        found = unpack(
            [os.path.join(dest, n) for n in os.listdir(dest) if n.lower().endswith(ARCHIVES)],
            dest,
        )
    if found:
        DATASET_DIR = dest

# ---- report ----------------------------------------------------------------
if not found:
    print("\nnothing usable / ничего подходящего:")
    if os.path.isdir(root):
        listing(root)
    raise SystemExit(
        "Attach a dataset with Add Input, or set AUDIO_URL in cell 1 to a direct link.\n"
        "Подключите датасет через Add Input или укажите прямую ссылку в AUDIO_URL (ячейка 1)."
    )

total_mb = sum(os.path.getsize(p) for p in found) / 1024 / 1024
print(f"\nusing / используется: {DATASET_DIR}")
print(f"{len(found)} file(s), {total_mb:.1f} MB / файлов: {len(found)}, {total_mb:.1f} МБ")

In [ ]:
#@title 4. Install dependencies / Установка зависимостей (5-10 min, once per session)
import os, subprocess

if "PY" not in globals() or "VENV_DIR" not in globals():
    raise SystemExit(
        "Run cell 1 (Settings) first: this cell needs PY and VENV_DIR.\n"
        "Сначала выполните ячейку 1 (Настройки) - в ней задаются пути."
    )

def sh(cmd, **kw):
    print("+", " ".join(cmd) if isinstance(cmd, list) else cmd, flush=True)
    subprocess.run(cmd, check=True, **kw)

subprocess.run(["pip", "install", "-q", "uv"], check=True)
os.makedirs(WORK, exist_ok=True)

if not os.path.exists(PY):
    sh(["uv", "venv", "--python", "3.11", VENV_DIR])

def uv(*args):
    sh(["uv", "pip", "install", "--python", PY, *args])

# torch 2.4.1 ships CUDA 12.1 wheels on PyPI and is known-good with this fairseq fork.
uv("torch==2.4.1", "torchaudio==2.4.1")
uv(
    "numpy==1.26.4", "scipy", "librosa==0.10.2", "soundfile", "numba", "llvmlite",
    "faiss-cpu", "praat-parselmouth", "scikit-learn",
    # matplotlib < 3.10: 3.10+ removed FigureCanvasAgg.tostring_rgb(), which the vendored
# RVC training code calls. The repo patch fixes it too; the pin is a second safety net.
    "tensorboard", "tensorboardX", "tqdm", "matplotlib<3.10", "PyYAML", "json5",
    "Cython", "ninja", "wheel", "setuptools==79.0.1", "packaging",
    "av", "ffmpeg-python", "matplotlib-inline", "omegaconf", "hydra-core>=1.3.2", "sacrebleu>=1.4.12",
)
# pyworld has no wheels for every Python build -> compile it against the venv numpy.
uv("--no-build-isolation", "pyworld")

# fairseq: clone the fork and build it against the installed torch.
# setup.py hardcodes -std=c++11 on Linux, which modern torch headers reject -> patch to C++17.
FAIRSEQ_DIR = os.path.join(WORK, "fairseq")
if not os.path.isdir(FAIRSEQ_DIR):
    sh(["git", "clone", "--depth", "1", "https://github.com/One-sixth/fairseq.git", FAIRSEQ_DIR])

setup_py = os.path.join(FAIRSEQ_DIR, "setup.py")
with open(setup_py) as f:
    setup_src = f.read()
if '"-std=c++11"' in setup_src:
    with open(setup_py, "w") as f:
        f.write(setup_src.replace('"-std=c++11"', '"-std=c++17"'))
    print("patched fairseq setup.py: -std=c++11 -> -std=c++17")

# Keep the build CPU/C++ only: a visible CUDA_HOME makes fairseq compile .cu files
# against Colab's toolkit, which often mismatches the pinned torch version.
build_env = {
    k: v for k, v in os.environ.items()
    if k not in ("CUDA_HOME", "CUDA_PATH", "TORCH_CUDA_ARCH_LIST")
}
sh(["uv", "pip", "install", "--python", PY, "--no-build-isolation", FAIRSEQ_DIR], env=build_env)

# Verify every import separately: one broken module must not hide the others.
MODULES = [
    "torch", "torchaudio", "fairseq", "faiss", "librosa",
    "pyworld", "parselmouth", "numba", "soundfile", "av", "pkg_resources",
]
failed = []
print("import check / проверка импортов:")
for mod in MODULES:
    r = subprocess.run(
        [PY, "-c", f"import {mod}; print(getattr({mod}, '__version__', 'ok'))"],
        capture_output=True, text=True,
    )
    if r.returncode == 0:
        print(f"  {mod:14s} OK   {r.stdout.strip()}")
    else:
        failed.append(mod)
        last = (r.stderr.strip().splitlines() or ["unknown error"])[-1]
        print(f"  {mod:14s} FAIL {last}")

if failed:
    print()
    print(">>> Fix the failed module(s) before continuing / исправьте перед продолжением:", ", ".join(failed))
    if "pkg_resources" in failed or "pyworld" in failed:
        print(f'    uv pip install --python {PY} "setuptools<80"')

In [ ]:
#@title 5. Get the code and the required weights / Код и веса
import os, subprocess

def sh(cmd, **kw):
    print("+", " ".join(cmd) if isinstance(cmd, list) else cmd, flush=True)
    subprocess.run(cmd, check=True, **kw)

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    sh(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR])
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)

BASE = "https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/"
FILES = {
    "hubert_base.pt":  "assets/hubert/hubert_base.pt",
    "rmvpe.pt":        "assets/rmvpe/rmvpe.pt",
}
if VERSION == "v2":
    FILES["pretrained_v2/f0G48k.pth"] = "assets/pretrained_v2/f0G48k.pth"
    FILES["pretrained_v2/f0D48k.pth"] = "assets/pretrained_v2/f0D48k.pth"
else:
    FILES[f"pretrained/f0G{SAMPLE_RATE}.pth"] = f"assets/pretrained/f0G{SAMPLE_RATE}.pth"
    FILES[f"pretrained/f0D{SAMPLE_RATE}.pth"] = f"assets/pretrained/f0D{SAMPLE_RATE}.pth"

for remote, local in FILES.items():
    dst = os.path.join(REPO_DIR, local)
    if os.path.exists(dst) and os.path.getsize(dst) > 1024:
        print("cached:", local)
        continue
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    sh(["curl", "-L", "--fail", "--retry", "3", "-o", dst, BASE + remote])

print("weights ready / веса готовы")

In [ ]:
#@title 6. Copy the dataset to fast local disk / Копирование датасета
# Kaggle input is read-only and mounted over the network: copying is the only
# way to get fast repeated reads. This cell prints one line per file and
# resumes from a partial copy.
# Вход Kaggle только для чтения и смонтирован по сети: копия нужна для быстрых
# повторных чтений. Ячейка печатает строку на каждый файл и продолжает с места
# обрыва.
import os, shutil, time

if not os.path.isdir(DATASET_DIR):
    raise SystemExit(
        f"Dataset folder not found: {DATASET_DIR}\n"
        "Put your audio there (WAV/MP3/FLAC/M4A/OGG). / Положите аудио в эту папку."
    )

EXTS = (".wav", ".mp3", ".flac", ".m4a", ".ogg", ".aif", ".aiff")
os.makedirs(LOCAL_DATA, exist_ok=True)

print(f"scanning {DATASET_DIR} ...", flush=True)
src_files = []
for root, _dirs, names in os.walk(DATASET_DIR):
    for name in sorted(names):
        if name.lower().endswith(EXTS):
            src_files.append(os.path.join(root, name))
print(f"found {len(src_files)} audio file(s) / найдено файлов: {len(src_files)}", flush=True)
if not src_files:
    raise SystemExit("No audio files found. / Аудиофайлы не найдены.")

started = time.time()
total = 0
for i, src in enumerate(src_files, 1):
    name = os.path.basename(src)
    dst = os.path.join(LOCAL_DATA, name)
    src_size = os.path.getsize(src)
    if os.path.exists(dst) and os.path.getsize(dst) == src_size:
        total += src_size
        print(f"[{i}/{len(src_files)}] skip {name} (already copied)", flush=True)
        continue
    t0 = time.time()
    shutil.copy2(src, dst)
    dt = max(time.time() - t0, 1e-6)
    total += src_size
    print(
        f"[{i}/{len(src_files)}] {name}: "
        f"{src_size / 1024 / 1024:.1f} MB in {dt:.1f}s "
        f"({src_size / 1024 / 1024 / dt:.1f} MB/s)",
        flush=True,
    )

print(
    f"{len(src_files)} file(s), {total / 1024 / 1024:.1f} MB copied in "
    f"{(time.time() - started) / 60:.1f} min / файлов: {len(src_files)}, "
    f"{total / 1024 / 1024:.1f} МБ",
    flush=True,
)

In [ ]:
#@title 6b. Normalize and split into 10-minute parts / Нормализация и нарезка
# Why: RVC decodes the whole file into RAM before slicing, so one multi-hour
# file can kill the runtime (session crash = out of memory).
# Splitting with ffmpeg keeps memory flat and gives preprocessing a uniform
# mono 48 kHz input, whatever the source format was.
# Зачем: RVC декодирует файл целиком в память, поэтому многочасовой файл
# валит рантайм (падение сессии = не хватило памяти). Нарезка через ffmpeg
# держит память ровно и приводит всё к одному формату: моно 48 кГц.
import glob, os, subprocess, time

SEGMENT_SECONDS = 600  # 10 min per part / по 10 минут
DONE_DIR = "/kaggle/temp/rvc/.split_done"
os.makedirs(TRAIN_DATA, exist_ok=True)
os.makedirs(DONE_DIR, exist_ok=True)

EXTS = (".wav", ".mp3", ".flac", ".m4a", ".ogg", ".aif", ".aiff")
sources = [
    os.path.join(LOCAL_DATA, name)
    for name in sorted(os.listdir(LOCAL_DATA))
    if name.lower().endswith(EXTS)
]
if not sources:
    raise SystemExit(f"No audio in {LOCAL_DATA} / нет аудио в {LOCAL_DATA}")

started = time.time()
for i, src in enumerate(sources, 1):
    base = os.path.splitext(os.path.basename(src))[0]
    marker = os.path.join(DONE_DIR, base + ".done")
    if os.path.exists(marker):
        print(f"[{i}/{len(sources)}] skip {base} (already split)", flush=True)
        continue
    t0 = time.time()
    pattern = os.path.join(TRAIN_DATA, base + "_%03d.wav")
    cmd = [
        "ffmpeg", "-nostdin", "-hide_banner", "-loglevel", "error", "-y",
        "-i", src,
        "-ac", "1", "-ar", str(SR), "-c:a", "pcm_s16le",
        "-f", "segment", "-segment_time", str(SEGMENT_SECONDS),
        "-reset_timestamps", "1",
        pattern,
    ]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr.strip()[-1500:], flush=True)
        raise SystemExit(f"ffmpeg failed on {src} / ffmpeg не смог обработать {src}")
    open(marker, "w").close()
    parts = len(glob.glob(os.path.join(TRAIN_DATA, base + "_*.wav")))
    print(f"[{i}/{len(sources)}] {base}: {parts} part(s) in {time.time() - t0:.1f}s", flush=True)

parts = sorted(glob.glob(os.path.join(TRAIN_DATA, "*.wav")))
size_mb = sum(os.path.getsize(p) for p in parts) / 1024 / 1024
print(
    f"{len(parts)} part(s), {size_mb:.1f} MB in {TRAIN_DATA} / "
    f"частей: {len(parts)}, {size_mb:.1f} МБ",
    flush=True,
)
print(f"split took {(time.time() - started) / 60:.1f} min", flush=True)

In [ ]:
#@title 7. Preprocess: slice, resample, normalize / Нарезка и нормализация
# Plain subprocess, output is inherited: no threads, no captured pipes, nothing
# that can hide an error. A 10-minute part takes about 1-2 minutes on a T4.
# Обычный subprocess с наследованием вывода: без потоков и без перехвата, так
# что ошибки не теряются. 10-минутная часть на T4 обрабатывается 1-2 мин.
import os, subprocess, time

os.makedirs(LOG_DIR, exist_ok=True)
log_file = os.path.join(LOG_DIR, "preprocess.log")

n_input = len(os.listdir(TRAIN_DATA)) if os.path.isdir(TRAIN_DATA) else 0
print(f"input: {TRAIN_DATA} ({n_input} file(s))", flush=True)
if n_input == 0:
    raise SystemExit(
        "dataset_48k is empty - run cell 6b first / "
        "папка пуста, сначала выполните ячейку 6b"
    )

cmd = [
    PY, "infer/modules/train/preprocess.py",
    TRAIN_DATA, str(SR), "1", LOG_DIR, "True", "3.7",
]
print("+", " ".join(cmd), flush=True)
print("running - one '-> Success' line per part / по строке на каждую часть:", flush=True)

started = time.time()
rc = subprocess.call(cmd, cwd=REPO_DIR, env=dict(os.environ, PYTHONUNBUFFERED="1"))
print(f"exit code {rc}, {(time.time() - started) / 60:.1f} min", flush=True)

if rc != 0:
    print("\nlast lines of the log / последние строки лога:", flush=True)
    try:
        with open(log_file, errors="replace") as f:
            print("".join(f.readlines()[-40:]))
    except FileNotFoundError:
        print(f"no log file at {log_file}")
    print(f"input folder: {TRAIN_DATA}", flush=True)
    print("files there:", len(os.listdir(TRAIN_DATA)) if os.path.isdir(TRAIN_DATA) else "MISSING", flush=True)

for d in ("0_gt_wavs", "1_16k_wavs"):
    p = os.path.join(LOG_DIR, d)
    print(d, len(os.listdir(p)) if os.path.isdir(p) else 0, "files")

if os.path.isdir(os.path.join(LOG_DIR, "0_gt_wavs")):
    bad = [
        line for line in open(log_file, errors="replace").read().splitlines()
        if "->" in line and "Success" not in line
    ]
    if bad:
        print(f"\n{len(bad)} part(s) failed - first error / не удалось обработать {len(bad)}:")
        print("\n".join(bad[:10]))

In [ ]:
#@title 8. Extract F0 with RMVPE / Извлечение F0 (rmvpe)
import subprocess
cmd = [
    PY, "infer/modules/train/extract/extract_f0_rmvpe.py",
    "1", "0", "0", LOG_DIR, "True",
]
print("+", " ".join(cmd), flush=True)
subprocess.run(cmd, cwd=REPO_DIR, check=True, env=dict(os.environ, PYTHONUNBUFFERED="1"))
print("F0 done / F0 готово")

In [ ]:
#@title 9. Extract HuBERT features / Признаки HuBERT
import subprocess
cmd = [
    PY, "infer/modules/train/extract_feature_print.py",
    "cuda", "1", "0", "0", LOG_DIR, VERSION, "True",
]
print("+", " ".join(cmd), flush=True)
subprocess.run(cmd, cwd=REPO_DIR, check=True, env=dict(os.environ, PYTHONUNBUFFERED="1"))
print("features done / признаки готовы")

In [ ]:
#@title 10. Write config.json and filelist.txt / Конфиг и список файлов
import json, os

feat_name = "3_feature768" if VERSION == "v2" else "3_feature256"
gt_dir    = os.path.join(LOG_DIR, "0_gt_wavs")
feat_dir  = os.path.join(LOG_DIR, feat_name)
f0_dir    = os.path.join(LOG_DIR, "2a_f0")
f0nsf_dir = os.path.join(LOG_DIR, "2b-f0nsf")

config_src = os.path.join(
    REPO_DIR, "configs", "v2" if VERSION == "v2" else "v1", f"{SAMPLE_RATE}.json"
)
with open(config_src) as f:
    cfg = json.load(f)
with open(os.path.join(LOG_DIR, "config.json"), "w") as f:
    json.dump(cfg, f, indent=2)

def stems(directory, suffixes):
    out = set()
    for name in os.listdir(directory):
        for suf in suffixes:
            if name.endswith(suf):
                out.add(name[: -len(suf)])
                break
    return out

names = (
    stems(gt_dir, [".wav"])
    & stems(feat_dir, [".npy"])
    & stems(f0_dir, [".wav.npy"])
    & stems(f0nsf_dir, [".wav.npy"])
)
names = sorted(names)
if len(names) < 10:
    raise SystemExit(
        f"Only {len(names)} usable chunks. Check the dataset and re-run cells 7-9.\n"
        f"Всего {len(names)} фрагментов — проверьте датасет и повторите шаги 7-9."
    )

lines = [
    f"{gt_dir}/{n}.wav|{feat_dir}/{n}.npy|{f0_dir}/{n}.wav.npy|{f0nsf_dir}/{n}.wav.npy|{SPEAKER_ID}"
    for n in names
]
with open(os.path.join(LOG_DIR, "filelist.txt"), "w") as f:
    f.write("\n".join(lines))

total_sec = len(names) * 3.7
print(f"{len(names)} chunks / фрагментов (~{total_sec/60:.1f} min of audio)")
print("config.json + filelist.txt written / записаны")

In [ ]:
#@title 11. Resume if a previous run exists / Продолжить с прошлого запуска
import glob, os, shutil

os.makedirs(CKPT_BACKUP, exist_ok=True)
have_local = bool(glob.glob(os.path.join(LOG_DIR, "G_*.pth")))
saved = sorted(glob.glob(os.path.join(CKPT_BACKUP, "*.pth")))

if have_local:
    print("local checkpoints found, keeping them / локальные чекпоинты на месте")
elif saved:
    for p in saved:
        shutil.copy2(p, LOG_DIR)
    print(f"restored {len(saved)} checkpoint file(s) / восстановлено {len(saved)}")
else:
    print("no checkpoints yet: training will start from the pretrained model / начнём с претрейна")

In [ ]:
#@title 12. Train / Обучение (40-90 min on T4)
import glob, os, subprocess, time

os.makedirs(CKPT_BACKUP, exist_ok=True)
os.makedirs(WEIGHTS_DIR, exist_ok=True)

# Mirror checkpoints into /kaggle/working every 5 minutes: that folder is what
# gets saved with the version, so a crash does not lose the training.
sync = subprocess.Popen(
    ["bash", "-c",
     f'while true; do '
     f'cp -f "{LOG_DIR}"/G_*.pth "{LOG_DIR}"/D_*.pth "{CKPT_BACKUP}"/ 2>/dev/null; '
     f'cp -f "{WEIGHTS_DIR}"/{EXPERIMENT}_e*.pth "{CKPT_BACKUP}"/ 2>/dev/null; '
     f'sleep 300; done']
)

cmd = [
    PY, "infer/modules/train/train.py",
    "-e", EXPERIMENT,
    "-sr", SAMPLE_RATE,
    "-f0", "1",
    "-bs", str(BATCH_SIZE),
    "-g", "0",
    "-te", str(TOTAL_EPOCHS),
    "-se", str(SAVE_EVERY),
    "-pg", PRETRAIN_G,
    "-pd", PRETRAIN_D,
    "-l", "1",       # keep only the latest G/D (the synced copy is the safety net)
    "-c", "0",       # no GPU data caching
    "-sw", "1",      # also write inference-ready <name>_eXXX_sYYYY.pth
    "-v", VERSION,
]
print("+", " ".join(cmd), flush=True)

started = time.time()
rc = -1
try:
    # MPLBACKEND=Agg: the notebook kernel sets an inline backend that does not
    # exist inside the venv, and train.py imports matplotlib to log spectrograms.
    # MPLBACKEND=Agg: ядро ноутбука задаёт inline-бэкенд, которого нет в venv,
    # а train.py импортирует matplotlib для спектрограмм.
    rc = subprocess.call(
        cmd, cwd=REPO_DIR,
        env=dict(os.environ, PYTHONUNBUFFERED="1", MPLBACKEND="Agg"),
    )
finally:
    sync.terminate()

# train.py exits with os._exit(2333333) when it finishes on purpose -> 149 in a shell.
if rc not in (0, 149, 2333333):
    raise SystemExit(f"Training failed with exit code {rc} / обучение упало с кодом {rc}")

made = sorted(glob.glob(os.path.join(WEIGHTS_DIR, f"{EXPERIMENT}_e*.pth")))
if not made:
    raise SystemExit(
        "Training exited without writing a single checkpoint - the real error is "
        "in the log above.\n"
        "Обучение завершилось, не записав ни одного чекпоинта - настоящая ошибка "
        "в логе выше."
    )

print(f"training finished in {(time.time() - started) / 60:.1f} min / обучение завершено")
print("inference checkpoints:")
for p in sorted(glob.glob(os.path.join(WEIGHTS_DIR, f"{EXPERIMENT}_e*.pth"))):
    print(" ", os.path.basename(p), f"{os.path.getsize(p)/1024/1024:.0f} MB")

In [ ]:
#@title 13. Build the faiss index / Построение индекса
# faiss lives in the venv, not in the notebook kernel, so the index is built by a
# short script executed with the venv interpreter.
# faiss стоит в venv, а не в ядре ноутбука, поэтому индекс строит отдельный
# скрипт, запущенный интерпретатором venv.
import glob, os, subprocess

dim = 768 if VERSION == "v2" else 256
feat_dir = os.path.join(LOG_DIR, "3_feature" + str(dim))

script = f"""
import glob, os
import faiss
import numpy as np

feat_dir = {feat_dir!r}
log_dir = {LOG_DIR!r}
experiment = {EXPERIMENT!r}
version = {VERSION!r}
dim = {dim}

npys = [np.load(p) for p in sorted(glob.glob(os.path.join(feat_dir, "*.npy")))]
if not npys:
    raise SystemExit("No features found / признаки не найдены")
big_npy = np.concatenate(npys, 0)
np.random.shuffle(big_npy)
print("features:", big_npy.shape, flush=True)

if big_npy.shape[0] > 2e5:
    from sklearn.cluster import MiniBatchKMeans
    centers = min(10000, big_npy.shape[0])
    print("kmeans ->", centers, "centers (a few minutes)", flush=True)
    big_npy = MiniBatchKMeans(
        n_clusters=centers, batch_size=1024, compute_labels=False,
        init="random", verbose=0,
    ).fit(big_npy).cluster_centers_
    print("after kmeans:", big_npy.shape, flush=True)

n_ivf = min(int(16 * np.sqrt(big_npy.shape[0])), big_npy.shape[0] // 39)
n_ivf = max(n_ivf, 1)
index = faiss.index_factory(dim, "IVF" + str(n_ivf) + ",Flat")
faiss.extract_index_ivf(index).nprobe = 1
index.train(big_npy.astype(np.float32))
index.add(big_npy.astype(np.float32))

name = "added_IVF" + str(n_ivf) + "_Flat_nprobe_1_" + experiment + "_" + version + ".index"
path = os.path.join(log_dir, name)
faiss.write_index(index, path)
print("index:", name, round(os.path.getsize(path) / 1024 / 1024), "MB", flush=True)
"""

print("building the index / строим индекс", flush=True)
rc = subprocess.call(
    [PY, "-c", script], cwd=REPO_DIR,
    env=dict(os.environ, MPLBACKEND="Agg"),
)
if rc != 0:
    raise SystemExit(f"index build failed with exit code {rc} / индекс не построился")
indexes = sorted(glob.glob(os.path.join(LOG_DIR, "added_*.index")))
print("index file(s):", [os.path.basename(p) for p in indexes])

In [ ]:
#@title 14. Export .pth + .index into /kaggle/working / Экспорт в /kaggle/working
# Everything written here ends up in the notebook Output (and in the version
# output after "Save Version"). / Всё отсюда попадает в Output ноутбука.
import glob, os, shutil, zipfile

os.makedirs(OUTPUT_DIR, exist_ok=True)
exported = []

models = sorted(glob.glob(os.path.join(WEIGHTS_DIR, f"{EXPERIMENT}_e*.pth")))
if not models:
    # fall back to the final checkpoint written by savee()
    models = sorted(glob.glob(os.path.join(WEIGHTS_DIR, f"{EXPERIMENT}*.pth")))
for p in models:
    shutil.copy2(p, OUTPUT_DIR)
    exported.append(os.path.basename(p))

for p in glob.glob(os.path.join(LOG_DIR, "added_*.index")):
    shutil.copy2(p, OUTPUT_DIR)
    exported.append(os.path.basename(p))

print(f"exported {len(exported)} file(s) to {OUTPUT_DIR} / файлов выгружено: {len(exported)}")
for name in exported:
    print(" ", name, f"{os.path.getsize(os.path.join(OUTPUT_DIR, name)) / 1024 / 1024:.0f} MB")

if exported:
    zip_path = os.path.join(OUTPUT_ROOT, f"{EXPERIMENT}_model.zip")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        for name in exported:
            z.write(os.path.join(OUTPUT_DIR, name), name)
    print(f"\nzip ready / архив готов: {zip_path} ({os.path.getsize(zip_path) / 1024 / 1024:.0f} MB)")
    print("Download it from the Output tab / скачайте на вкладке Output.")

print()
print("Next / Далее: install the ZIP with 'Install model from ZIP' in RVC-WebUI-MacOS,")
print("or copy the .pth to assets/weights/ and the .index to logs/.")

## After training / После обучения

**1. Download / Скачать.** Cell 14 packs everything into `/kaggle/working/rvc/<name>_model.zip`. Open the **Output** tab of the notebook (or of the finished version) and download it. The separate `.pth` and `.index` sit next to it in `rvc/<name>/`.

Ячейка 14 складывает всё в `/kaggle/working/rvc/<имя>_model.zip`. Откройте вкладку **Output** (ноутбука или завершённой версии) и скачайте архив. Отдельные `.pth` и `.index` лежат рядом в `rvc/<имя>/`.

**2. Install on the Mac / Установить на Mac.** In RVC-WebUI-MacOS, open the **Install model from ZIP (.pth + .index)** panel, drop in that ZIP, click **Install**. Manual fallback: `.pth` → `assets/weights/`, `.index` → `logs/`.

**3. Inference settings for speech / Настройки для речи**

| Setting | Value |
| --- | --- |
| F0 method | `rmvpe` |
| Pitch | `0` (or ±12 to change gender / ±12 для смены пола) |
| Index rate | `0.5–0.75` |
| Protect | `0.33` |
| RMS mix | `0.25` |

**4. Listen critically / Слушать критически.** Test on speech that was **not** in the dataset. Muffled - use an earlier epoch. Unstable - train 50-100 more epochs.

Проверяйте на речи, которой **не было** в датасете. Глухо - возьмите более раннюю эпоху. Нестабильно - добавьте 50-100 эпох.

## Troubleshooting / Если что-то пошло не так

| Problem | Fix |
| --- | --- |
| `No dataset attached` / `No audio files under /kaggle/input` | Use **Add Input** in the right sidebar and attach your dataset. |
| `pip`/`uv` cannot download anything | **Settings → Internet → On**, then re-run cell 4. |
| `CUDA out of memory` | Lower `BATCH_SIZE` to 6 or 4 and re-run cell 12. |
| `pyworld` fails with `No module named 'pkg_resources'` | setuptools 82+ removed it. `uv pip install --python /kaggle/temp/rvc/.venv/bin/python "setuptools<80"`, then re-run cell 4. |
| `fairseq` build fails | Re-run cell 4 (it patches `-std=c++11` to C++17). If it still fails, **Factory reset** the session and run 4 again. |
| Interactive session was killed / disconnected | Re-run cells 1-14. Training resumes from `/kaggle/working/rvc/<name>/checkpoints` if a version was saved, otherwise from the pretrained model. |
| `GPU quota exceeded` | Free accounts get ~30 GPU-hours per week; wait for the weekly reset or use a CPU session for preprocessing. |
| `/kaggle/working` is full | Only 20 GB are saved. A model is ~110 MB - delete old versions or unused outputs. |
| Cell 6 seems to hang | Kaggle input is read-only and slow. The cell prints one line per file and resumes from a partial copy. |
| Cell 7 prints nothing for minutes | Preprocess prints one `-> Success` line per 10-minute part, so 1-2 minutes of silence is normal. Nothing after 5 minutes - check `logs/<name>/preprocess.log`. |
| Preprocessing is slow | It runs one process on purpose (fork after numba/librosa can deadlock). For speed edit cell 7: `"1"` -> `"4"` and `True` -> `"False"`. |
| `Only N usable chunks` | The dataset is too short or has too much silence. Add clean speech and re-run 6, 6b, then 7-10. |
| Model sounds metallic | Fewer epochs, cleaner source, no aggressive denoise. Try `Index rate` 0.3-0.5 at inference. |

---

If this notebook helped, you can support the project / если ноутбук помог - можно поддержать проект: [PayPal](https://www.paypal.com/donate/?business=0eskimo0%40gmail.com&no_recurring=0&item_name=Support+RVC-WebUI-MacOS&currency_code=USD)